In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")

In [ ]:
from torchvision.utils import draw_bounding_boxes
from pathlib import Path
import json
import shutil
from torchvision.io import read_image, write_jpeg
import matplotlib.pyplot as plt

from colony_segmentation.dataset_transforms import get_transforms
from colony_segmentation.model import (
    get_model_instance_segmentation,
)


num_classes = 2

eval_transform = get_transforms(is_train=False)


def threshold_predictions(pred, confidence_threshold):
    # Create a mask for scores above the threshold
    score_mask = pred["scores"] > confidence_threshold

    # Filter the labels and boxes using the mask
    filtered_labels = [label for label, keep in zip(pred["labels"], score_mask) if keep]
    filtered_boxes = pred["boxes"][score_mask]

    new_preds = {
        "labels": filtered_labels,
        "scores": pred["scores"][score_mask],
        "boxes": filtered_boxes,
    }

    if "masks" in pred:
        new_preds["masks"] = pred["masks"][score_mask]

    return new_preds


def draw_boxes_on_image(
    image: torch.Tensor, boxes: torch.Tensor, scores: torch.Tensor = None
):
    image = draw_bounding_boxes(
        image,
        boxes,
        colors="#ff1111",
        width=8,
        font_size=52,
        font=str(Path("./assets/zed-mono-bold.ttf")),
    )

    return image


def predict(model_path: str, image: torch.Tensor, confidence_threshold=0):
    model = get_model_instance_segmentation(num_classes).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    pred = threshold_predictions(pred, confidence_threshold)

    return image, pred["boxes"].long(), pred["scores"]


def load_ground_truth(image_path: Path):
    image = read_image(str(image_path))

    with open(image_path.parents[1] / "boxes" / f"{image_path.stem}.json") as f:
        ground_truth = json.load(f)

    # convert boxes from xywh dict to xyxy tuple
    boxes = [
        [box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]]
        for box in ground_truth["boxes"]
    ]

    boxes = torch.tensor(boxes)

    return image, boxes


images_path = Path("val_data/images")

image_paths = [
    images_path / "Y_500µl_2_default_ gvpcagar_default_15122023144938.jpg",
    images_path / "default_mmrsbpb_default_19012024141840.jpg",
    images_path
    / "rod_lentilactobacillus_buchneri_20_default_mmrsbpb_3_07102022120738.jpg",
    images_path / "default_mmrsbpb_default_19012024140805.jpg",
]

for image_path in image_paths:
    # Copy image to current directory
    destination_path = Path("./") / image_path.name

    shutil.copy(image_path, destination_path)

    image, boxes = load_ground_truth(image_path)
    _, pred_boxes, pred_scores = predict(
        "anchors.pth",
        image,
        0,
    )

    print("Ground Truth Boxes: ", len(boxes))

    gt_image = draw_boxes_on_image(image, boxes)
    pred_image = draw_boxes_on_image(image, pred_boxes, pred_scores)

    # Save images
    write_jpeg(gt_image, f"gt_{image_path.stem}.jpg")
    write_jpeg(pred_image, f"pred_{image_path.stem}.jpg")

    # Display images

    plt.figure(figsize=(10, 10))
    plt.imshow(gt_image.permute(1, 2, 0).numpy())
    plt.title("Ground Truth")
    plt.show()

    plt.figure(figsize=(10, 10))
    plt.imshow(pred_image.permute(1, 2, 0).numpy())
    plt.title("Predictions")
    plt.show()